# 01 · Ingestion and cleaning

Half-hourly electricity readings from the London households in UK Power Networks' **Low Carbon London** trial (November 2011 to February 2014): 112 CSV files and about 168 million rows. This notebook turns them into a table of complete household-days, with every repaired or suspicious value flagged rather than silently changed.

| Stage | Rows | Household-days |
|---|---:|---:|
| Raw half-hourly files | 167,817,021 | |
| Off-grid event records removed | −5,560 | |
| On the half-hour grid | 167,811,461 | 3,510,403 |
| Incomplete days dropped | −294,249 | −20,023 |
| One-slot outages repaired (imputed) | +21,028 | |
| **Cleaned table** | **167,538,240** | **3,490,380** (each exactly 48 readings) |

Flags kept in the output: `imputed` (21,028 readings), `suspect_spike` (9 readings), `zero_day` (14,954 household-days).

**To run:** Google Colab, with Kaggle API credentials in Colab Secrets as `KAGGLE_USER` and `KAGGLE_API_TOKEN` (notebook access switched on).

## Setup

In [ ]:
!pip install pyspark

import os
from google.colab import userdata
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window 
from pyspark.sql import types as T
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import DoubleType

In [ ]:
#@title  Download the dataset

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')




# Create the standard project folders
!mkdir -p data src notebooks
print("Project folders 'data/', 'src/', and 'notebooks/' are ready.")




# Download the dataset directly into the data folder and unzip it
!kaggle datasets download -d jeanmidev/smart-meters-in-london -p data --unzip
print("Dataset downloaded and extracted into the 'data/' folder!")




!ls -l data

## 1. Load the data and check the schema

I let Spark infer the schema first, to see what it makes of the raw files. It reads `tstp` correctly as a timestamp, but the energy column comes back as **`string`**.

The reason: missing readings are stored as the literal text `Null`. A column holding both numbers and words falls back to string, and a plain null check on it reports **zero** missing values, because `"Null"` is a valid string. The data looks complete when it isn't.

In [ ]:
#@title Load the details and check the schema


# Initialize your session and load with header configuration
spark = SparkSession.builder.appName('Dataframe').getOrCreate()


halfHour_df = spark.read.csv('data/halfhourly_dataset/halfhourly_dataset', header=True, inferSchema=True)


# Display the top row to verify
display(halfHour_df.head())
halfHour_df.show(3)


halfHour_df.printSchema()

## 2. Fix the energy column type

Before the cast there are no real nulls. `try_cast` converts every valid number to a double and turns the text `Null` into a real null without raising an error (a plain `.cast()` fails on it in ANSI mode). After the cast the column has **5,560 nulls**, matching the count of text `Null` values, so nothing else was malformed.

In [ ]:
#@title Adjusting schema issues


# 1. Count actual system nulls vs text "Null" strings

halfHour_df.select(
    count(when(col("energy(kWh/hh)").isNull(), 1)).alias("System_Null_Count"),
    count(when(col("energy(kWh/hh)") == "Null", 1)).alias("Text_Null_Count"),
    count(when(col("energy(kWh/hh)") == "", 1)).alias("Text_emplty_Count")
).show()


# Cast column to double and convert text "Null" to true nulls
# try_cast automatically forces proper numbers to Double and turns text like 'Null' into standard NULLs safely
numeric_halfhourly_df = halfHour_df.withColumn(
    "energy(kWh/hh)",
    col("energy(kWh/hh)").try_cast(DoubleType()))

# Verify counts again
numeric_halfhourly_df.select(
    count(when(col("energy(kWh/hh)").isNull(), 1)).alias("System_Null_Count"),
).show()

numeric_halfhourly_df.printSchema()

In [ ]:
#@title Check null values

#drop column
#drop rows
#Filling missing values with mean, median, mode


# Count how many nulls exist in each column of the massive daily dataset
numeric_halfhourly_df.select([count(when(col(c).isNull(), c)).alias(c) for c in numeric_halfhourly_df.columns]).show()

## 3. Where are the nulls?

Before deciding to drop or fill, I looked at where the 5,560 nulls sit.

- **5,521 of them fall on one afternoon:** 18 December 2012, between 15:13 and 15:17, one per household, across almost the whole meter fleet.
- **None of them are half-hourly readings.** Real readings sit exactly on :00 or :30 with zero seconds. All 5,560 nulls have off-grid, seconds-precise timestamps (15:13:44, 15:17:02, …), including the 39 scattered across other dates.
- **The regular readings are all there.** 5,523 households have both their 15:00 and 15:30 readings that day, so the event records sat alongside normal data rather than replacing it.
- **Off-grid and null are the same set of rows.** No off-grid row has a value, and no on-grid row is null.

So the dataset has no missing values in the usual sense. The nulls are spurious event records, probably from a fleet-wide system event (the cause isn't documented in the data). One rule removes them and explains why: **keep only readings on the half-hour grid.** No real data is lost.

In [ ]:
#@title A quick look at where the nulls are


Null = numeric_halfhourly_df.filter(col("energy(kWh/hh)").isNull())
Null.show()

Null.groupBy("LCLid").count().show()

Null.groupBy(F.to_date("tstp").alias("day")).count().show()

In [ ]:
df = numeric_halfhourly_df
on_grid = F.minute("tstp").isin(0, 30) & (F.second("tstp") == 0)
is_null = F.col("`energy(kWh/hh)`").isNull()

# 1. Are the nulls on-grid or off-grid?
df.filter(is_null).groupBy(on_grid.alias("on_grid")).count().show()

# 2. Do the households still have their real 15:00 and 15:30 readings that day?
df.filter((F.to_date("tstp") == "2012-12-18") & (F.hour("tstp") == 15) & on_grid) \
  .groupBy(F.date_format("tstp", "HH:mm").alias("slot")).count().show()



# Check with other time stamps
df.filter((F.to_date("tstp") == "2012-12-18") & (F.hour("tstp") == 14) & on_grid) \
  .groupBy(F.date_format("tstp", "HH:mm").alias("slot")).count().show()

# 3. Are there off-grid rows that DO have a value?
df.filter(~on_grid & ~is_null).count()

In [ ]:
grid_halfhourly_df = numeric_halfhourly_df.filter(on_grid)


# Double-check if any null values remain on our grid
remaining_nulls = grid_halfhourly_df.filter(F.col("`energy(kWh/hh)`").isNull()).count()
total_rows_left = grid_halfhourly_df.count()

print(f"-> Remaining null values: {remaining_nulls}")
print(f"-> Cleaned grid dataset total row count: {total_rows_left}")

Every cell so far rereads all 112 CSV files. Writing the on-grid data to Parquet once makes everything after this much faster, and renaming the columns here removes the need for backticks around `energy(kWh/hh)`.

In [ ]:
(grid_halfhourly_df
   .withColumnRenamed("LCLid", "household_id")
   .withColumnRenamed("tstp", "timestamp")
   .withColumnRenamed("energy(kWh/hh)", "energy_kwh")
   .write.mode("overwrite").parquet("data/parquet/halfhourly_clean"))

hh = spark.read.parquet("data/parquet/halfhourly_clean")

## 4. Duplicates

No household has two readings for the same timestamp. This is also a check on clock changes: had the timestamps been UK local time, the repeated hour in late October would show up as duplicates here.

In [ ]:
#@title Check for duplicates


dupes = hh.groupBy("household_id", "timestamp").count().filter("count > 1")
print(f"Duplicate (household, timestamp) pairs: {dupes.count():,}")
dupes.show()

## 5. How complete is each household-day?

A null check can't see the commonest kind of gap in meter data: readings that were never recorded, so the row simply isn't there. Counting readings per household per day against the expected 48 finds them.

98.8% of household-days are complete. The rest aren't one smooth spread of gaps; there are three distinct features, so I checked whether each partial day sits at the start or end of a household's time in the trial, what time single readings fall at, and which dates the 47- and 46-reading days fall on.

There are no 50-reading days, and the 46-reading days don't cluster in late March, so clock changes need no special handling.

In [ ]:
#@title Check how complete each household-day is 


per_day = hh.groupBy("household_id", F.to_date("timestamp").alias("day")).count()
per_day.groupBy("count").agg(F.count("*").alias("household_days")).orderBy("count").show(60)

In [ ]:
hh_d = hh.withColumn("day", F.to_date("timestamp"))
per_day = hh_d.groupBy("household_id", "day").agg(F.count("*").alias("n"))

# 1. Are partial days at each household's edges (first/last day) or in the middle?
bounds = per_day.groupBy("household_id").agg(F.min("day").alias("first_day"),
                                             F.max("day").alias("last_day"))
pos = (per_day.join(bounds, "household_id")
       .withColumn("position", F.when(F.col("day") == F.col("first_day"), "first")
                                .when(F.col("day") == F.col("last_day"), "last")
                                .otherwise("interior"))
       .withColumn("bucket", F.when(F.col("n") == 48, "48")
                              .when(F.col("n") == 47, "47")
                              .when(F.col("n") == 1, "1")
                              .otherwise("2-46")))
pos.groupBy("bucket", "position").count().orderBy("bucket", "position").show()

# 2. The single-reading days: what time is that one reading?
singles = hh_d.join(per_day.filter("n = 1"), ["household_id", "day"])
singles.groupBy(F.date_format("timestamp", "HH:mm").alias("time")).count() \
       .orderBy(F.desc("count")).show(5)

# 3. The 47- and 46-reading days: concentrated on particular dates?
for k in (47, 46):
    print(f"Days with {k} readings, top dates:")
    per_day.filter(f"n = {k}").groupBy("day").count().orderBy(F.desc("count")).show(10)

### What the diagnostics show, and what I did with each group

| Day type | Household-days | What it is | Action |
|---|---:|---|---|
| Complete (48 readings) | 3,469,352 | Normal data, including 121 complete last days | Keep |
| Interior, 47 readings | 21,028 | Fleet-wide one-slot outages, clustered on dates (826 households on 17 Jun 2012, 600 on 14 Jun 2012, 569 on 20 Jan 2014) | Repair the slot, flag as `imputed` |
| Partial first days | 5,561 | The meter joined the trial partway through a day | Drop |
| Last day, single 00:00 reading | 5,437 (+3 partial) | A stray midnight reading after the final full day | Drop |
| Interior, single 00:00 reading | 5,864 | An orphan midnight reading at the start of an outage | Drop |
| Interior, 2–46 readings | 3,158 | Genuine gaps, spread across dates | Drop |

Two decisions are worth explaining.

**Why drop incomplete days instead of the missing readings alone?** Summing a day with missing half-hours understates its consumption, and nothing warns you. Filling gaps with a column mean is worse: it puts the same invented value into a 3am slot and a 6pm slot, for a low user and for a household with electric heating alike.

**Why repair the 47-reading days instead of dropping them?** They're concentrated on specific dates. Dropping them would remove up to 826 households from a single day, which would bias anything analysed by date, including the weather analysis. One interpolated half-hour out of 48, taken from the neighbouring readings in the same household and day, is a small and defensible repair, and the `imputed` flag lets later analyses exclude it.

In [ ]:
days = (per_day.join(bounds, "household_id")
        .withColumn("is_edge", (F.col("day") == F.col("first_day")) | (F.col("day") == F.col("last_day"))))

keep_days = days.filter(F.col("n") == 48).select("household_id", "day")
fix_days  = days.filter((F.col("n") == 47) & ~F.col("is_edge")).select("household_id", "day")

# Complete days, as they are
complete = (hh_d.join(keep_days, ["household_id", "day"])
            .select("household_id", "timestamp", "energy_kwh")
            .withColumn("imputed", F.lit(False)))

# 47-reading days: build the full 48-slot grid, then fill the gap
grid = fix_days.withColumn("timestamp", F.expr(
    "explode(sequence(timestamp(day), timestamp(day) + INTERVAL 23 HOURS 30 MINUTES, INTERVAL 30 MINUTES))"))

w = Window.partitionBy("household_id", "day").orderBy("timestamp")
filled = (grid.join(hh_d.select("household_id", "timestamp", "energy_kwh"),
                    ["household_id", "timestamp"], "left")
          .withColumn("prev", F.last("energy_kwh", ignorenulls=True).over(w.rowsBetween(Window.unboundedPreceding, -1)))
          .withColumn("next", F.first("energy_kwh", ignorenulls=True).over(w.rowsBetween(1, Window.unboundedFollowing)))
          .withColumn("imputed", F.col("energy_kwh").isNull())
          .withColumn("energy_kwh", F.coalesce("energy_kwh", (F.col("prev") + F.col("next")) / 2, "prev", "next"))
          .select("household_id", "timestamp", "energy_kwh", "imputed"))

final = complete.unionByName(filled)
final.write.mode("overwrite").parquet("data/parquet/halfhourly_complete_days")

The check: every household-day should now have exactly 48 readings, and there should be 21,028 imputed readings. The row count confirms it independently: (3,469,352 + 21,028) days × 48 = 167,538,240.

In [ ]:
final = spark.read.parquet("data/parquet/halfhourly_complete_days")
check = final.groupBy("household_id", F.to_date("timestamp").alias("day")).count()
check.groupBy("count").count().show()                       # should be a single row: 48
print(f"Imputed readings: {final.filter('imputed').count():,}")   # should be 21,028

## 6. Are the values plausible?

The checks so far were about which readings exist. These look at the numbers themselves.

- **No negative readings.**
- **The distribution is realistic.** Median 0.117 kWh per half-hour (about 230 W average draw), 99th percentile 1.49 kWh, 99.9th percentile 2.79 kWh.
- **The maximum, 10.76 kWh, looks like real heavy load.** That's about 21.5 kW for half an hour, near the limit of a standard 100 A single-phase supply. The top values come from two households and are sustained across consecutive half-hours and repeated on several winter evenings, which is how genuine heavy use (electric heating, for example) looks. Meter errors tend to be isolated one-off spikes.

To test for errors directly, I looked for isolated spikes: a high reading with low readings either side. There are 2,330 above 3 kWh, but they sit just above 3 kWh, the size of a high-power appliance such as an electric shower used for part of a half-hour, and some recur at the same time of day in the same household. Those are behaviour, not faults.

Above about 6 kWh the picture changes. There's a clear break in the distribution: the next isolated spike after 5.8 kWh is 8.3 kWh. **Nine readings** sit above it, eight from one household (MAC004488) at a near-constant 8.3–8.6 kWh, often in the early hours, with near-zero readings either side. The repeated, almost identical size makes these look like a meter artefact. They make up 67 kWh of that household's 8,330 kWh total (under 1%).

I can't prove they're errors, so I **flagged them as `suspect_spike` and kept them**. No values were capped: capping real high users would distort exactly the households that matter most for peak demand.

In [ ]:
#@title Check the values if they are pausible


final.agg(
    F.count("*").alias("rows"),
    F.sum((F.col("energy_kwh") < 0).cast("int")).alias("negative"),
    F.sum((F.col("energy_kwh") == 0).cast("int")).alias("zero"),
    F.min("energy_kwh").alias("min"),
    F.expr("percentile_approx(energy_kwh, array(0.5, 0.99, 0.999))").alias("p50_p99_p999"),
    F.max("energy_kwh").alias("max"),
).show(vertical=True, truncate=False)

final.orderBy(F.desc("energy_kwh")).show(10)   # who are the extreme values?

In [ ]:
w = Window.partitionBy("household_id").orderBy("timestamp")
spikes = (final
          .withColumn("prev", F.lag("energy_kwh").over(w))
          .withColumn("next", F.lead("energy_kwh").over(w))
          .filter((F.col("energy_kwh") > 3) & (F.col("prev") < 0.5) & (F.col("next") < 0.5)))
print(f"Isolated spikes (>3 kWh, neighbours <0.5): {spikes.count():,}")
spikes.show(10)

spikes.orderBy(F.desc("energy_kwh")).show(10)
spikes.select(F.max("energy_kwh")).show()

In [ ]:
w = Window.partitionBy("household_id").orderBy("timestamp")
final = (final
         .withColumn("prev", F.lag("energy_kwh").over(w))
         .withColumn("next", F.lead("energy_kwh").over(w))
         .withColumn("suspect_spike", F.coalesce(
             (F.col("energy_kwh") > 6) & (F.col("prev") < 0.5) & (F.col("next") < 0.5), F.lit(False)))
         .drop("prev", "next"))

final.filter("suspect_spike").groupBy("household_id").count().orderBy(F.desc("count")).show()

# How much of MAC004488's total consumption do the spikes make up?
final.filter(F.col("household_id") == "MAC004488").agg(
    F.sum("energy_kwh").alias("total_kwh"),
    F.sum(F.when(F.col("suspect_spike"), F.col("energy_kwh"))).alias("spike_kwh")).show()

## 7. Days with zero consumption

About 2 million readings (1.2%) are exactly zero. Scattered zeros are normal, since a low night-time draw can round down to zero. A **whole day** of zeros is different: it usually means a dead meter or an empty property.

There are **14,954 whole zero days**, and they're heavily concentrated. The ten worst households account for 5,513 of them (37%), and the worst, MAC000197, has 783. These look like meters that stopped recording consumption while still reporting.

I **flagged them as `zero_day`** rather than dropping them. Whether a household with long zero stretches stays in an analysis is a decision for the preparation stage, where the household-coverage threshold is set.

In [ ]:
daily = final.groupBy("household_id", F.to_date("timestamp").alias("day")) \
             .agg(F.sum("energy_kwh").alias("kwh"))
zero_days = daily.filter("kwh = 0")
print(f"Whole days at zero: {zero_days.count():,}")
zero_days.groupBy("household_id").count().orderBy(F.desc("count")).show(10)

In [ ]:
#@title Flag the zero days

print(f"Households with at least one zero day: {zero_days.select('household_id').distinct().count():,}")

zero_day_keys = zero_days.select("household_id", "day").withColumn("zero_day", F.lit(True))

final = (final
         .withColumn("day", F.to_date("timestamp"))
         .join(zero_day_keys, ["household_id", "day"], "left")
         .withColumn("zero_day", F.coalesce(F.col("zero_day"), F.lit(False)))
         .drop("day"))

## 8. Save the cleaned table

Expected: 167,538,240 rows, 21,028 imputed, 9 suspect spikes, and 717,792 zero-day rows (14,954 days × 48).

In [ ]:
#@title Save the cleaned dataset

final.write.mode("overwrite").parquet("data/parquet/halfhourly_cleaned")

cleaned = spark.read.parquet("data/parquet/halfhourly_cleaned")
cleaned.printSchema()
cleaned.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("imputed").cast("int")).alias("imputed"),
    F.sum(F.col("suspect_spike").cast("int")).alias("suspect_spike"),
    F.sum(F.col("zero_day").cast("int")).alias("zero_day_rows"),
).show()

## 9. Figure: what the gaps looked like

One chart summarises section 5. On a log scale, the three patterns behind the incomplete days stand out clearly against the 3.47 million complete days: the spike of single 00:00 readings, the hump of partial first days, and the bar of one-slot outages at 47.

In [ ]:
#@title Figure: readings per household-day

import matplotlib.pyplot as plt

# per_day comes from section 5: one row per household-day with its reading count n
dist = (per_day.groupBy("n").agg(F.count("*").alias("household_days"))
        .toPandas())          # only 43 rows, safe to bring to the driver

!mkdir -p figures

dist = dist.sort_values("n")
colour = {48: "#2b6a4f", 47: "#3b7dd8"}
bars = [colour.get(n, "#c9c4bb") for n in dist["n"]]

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.bar(dist["n"], dist["household_days"], color=bars, width=0.8)
ax.set_yscale("log")
ax.set_xlim(0, 49.5)
ax.set_ylim(0.7, 2e7)
ax.set_xticks([1, 8, 16, 24, 32, 40, 48])
ax.set_xlabel("Readings recorded in the day (48 expected)")
ax.set_ylabel("Household-days (log scale)")
ax.set_title("Readings per household-day, London smart meter trial", loc="left", fontsize=13)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

def note(x, y, text, tx, ty):
    ax.annotate(text, xy=(x, y), xytext=(tx, ty), fontsize=9, color="#333",
                arrowprops=dict(arrowstyle="-", color="#888", lw=0.8))

note(1, 11301, "1 reading, always 00:00\nstray reading after the last full day,\nor just before an outage (dropped)", 3.5, 4e5)
note(28, 543, "Partial first days:\nmeter joined mid-day (dropped)", 14, 8e3)
note(47, 21209, "47 readings: one-slot outages\nclustered on dates (21,028 repaired)", 30, 2.2e5)
note(40, 233, "Scattered genuine gaps\n(dropped)", 35.5, 2.2e3)
note(48, 3469352, "48 readings: 3,469,352\ncomplete days (kept)", 33, 5e6)

fig.text(0.01, 0.01, "Grey: dropped    Blue: one slot interpolated and flagged    Green: kept as recorded",
         fontsize=8.5, color="#555")
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig("figures/readings_per_household_day.png", dpi=160)
plt.show()


## Still open

- **Timestamp convention.** Every household's final reading falls at 00:00 on the day after its last full day, which suggests timestamps may mark the *end* of each half-hour rather than the start. It doesn't change any of the cleaning here, but it decides which day the 00:00 reading counts towards in daily totals. Notebook 02 settles this before building them.
- **Household coverage.** Some households have only a short run of complete days, or long zero stretches. The minimum to keep a household depends on the analysis (the tariff comparison needs data both before and during 2013), so that threshold is set in notebook 02.